In [2]:
import numpy as np

In [3]:
v = "How does approximate nearest neighbor search work?"

Q1

In [4]:
from embedder import Embedder

2026-06-24 12:42:30.821212233 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [5]:
embedder = Embedder()

Loaded model from models/Xenova/all-MiniLM-L6-v2


In [6]:
result = embedder.encode("How does approximate nearest neighbor search work?")

In [7]:
len(result)

384

In [8]:
result[0]

np.float64(-0.02058203437252893)

In [9]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [10]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

Q2

In [11]:
content = [p['content'] for p in documents if p['filename']=="02-vector-search/lessons/07-sqlitesearch-vector.md"]

In [12]:
content[0]

'# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done so far is ex

In [13]:
np.dot(embedder.encode(content[0]), result)

np.float64(0.36107026789538205)

Q3

In [14]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [15]:
len(chunks)

295

In [16]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [17]:
X =embedder.encode_batch([chunk['content'] for chunk in chunks])

In [18]:
X.shape

(295, 384)

In [19]:
scores = X.dot(result)

In [20]:
scores.shape

(295,)

In [21]:
scores

array([ 3.15187705e-01,  2.01479593e-01,  5.90559560e-02, -7.67661858e-02,
        1.18452480e-01, -1.41697042e-01, -2.81406552e-02, -4.65669225e-02,
       -2.06994704e-02, -6.07744087e-02,  2.13273853e-01,  8.87601799e-02,
       -1.97269351e-02,  3.11630016e-01,  5.51034674e-01,  2.04008048e-01,
        2.12515842e-01,  1.93649180e-01,  2.51961293e-01,  1.31078643e-01,
        2.59120579e-01,  7.63816008e-02,  9.59193707e-02,  9.81472975e-03,
       -3.59106882e-02,  1.01211577e-02,  1.10786937e-01, -9.90259208e-02,
       -3.71170151e-02,  7.59057570e-02, -3.35340540e-02,  8.86841309e-03,
        1.02636405e-01,  6.89614876e-02,  1.29408856e-01,  2.57709091e-01,
        3.23680614e-01,  1.06350075e-01,  5.61891367e-02,  2.34017457e-01,
        1.97954387e-01,  9.64296290e-02,  1.93709917e-01,  2.16719271e-01,
        3.48340456e-01,  5.10906092e-02,  2.05212837e-01,  1.05416170e-01,
       -3.25432514e-02,  4.94665548e-02,  2.38574865e-01, -3.44207108e-02,
        1.82165438e-01,  

In [22]:
chunks[np.argmax(scores)]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

Q4

In [23]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [24]:
v = "What metric do we use to evaluate a search engine?"

In [25]:
v_encoded = embedder.encode(v)

In [26]:
results = vindex.search(v_encoded, 
                        num_results=5)

In [27]:
results

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

Q5

In [28]:
v = "How do I store vectors in PostgreSQL?"

In [29]:
from minsearch import Index
index = Index(
        text_fields=["content"],
    )
index.fit(chunks)

In [32]:
[p['filename'] for p in index.search(v, num_results=5)]

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [33]:
v_encoded = embedder.encode(v)

In [34]:
[p['filename'] for p in vindex.search(v_encoded, num_results=5)]

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']